# 🚕 NYC Yellow Taxi Data Visualization

This notebook reads the aggregated output from our Spark Delta tables and generates visualizations for the final report.

In [ ]:
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

spark = SparkSession.builder \
    .appName("Visualization") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

### 1. Rolling 7-Day Average Taxi Demand

In [ ]:
# Load Delta table
rolling_avg_df = spark.read.format("delta").load("../data/processed/rolling_7_day_avg")
rolling_pd = rolling_avg_df.toPandas()
rolling_pd['date'] = pd.to_datetime(rolling_pd['date'])

plt.figure(figsize=(16, 7))
sns.lineplot(data=rolling_pd, x='date', y='rolling_7_day_avg', hue='pickup_borough', linewidth=2)
plt.title('Rolling 7-Day Average Taxi Demand by Borough', fontsize=18, fontweight='bold')
plt.xlabel('Date', fontsize=14)
plt.ylabel('Average Daily Trips', fontsize=14)
plt.legend(title='Borough')
plt.grid(True, alpha=0.3)
plt.show()

### 2. Top 3 Busiest Zones by Hour

In [ ]:
top_zones_df = spark.read.format("delta").load("../data/processed/top_3_zones_by_hour")
top_zones_pd = top_zones_df.toPandas()

plt.figure(figsize=(16, 7))
sns.scatterplot(data=top_zones_pd, x='hour', y='total_trips', hue='pickup_zone', size='rank', sizes=(200, 50), alpha=0.7)
plt.title('Top 3 Busiest Pickup Zones per Hour', fontsize=18, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=14)
plt.ylabel('Total Trips', fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()